In [63]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pycountry
ROUNDS_URL = "https://raw.githubusercontent.com/pedrosalgado22/Live_Geoguessr_Dashboard/main/roundsfinal.csv"
GAMES_URL = "https://raw.githubusercontent.com/pedrosalgado22/Live_Geoguessr_Dashboard/main/games.csv"

rounds = pd.read_csv(ROUNDS_URL)
games = pd.read_csv(GAMES_URL)

# Type normalization for downstream visualizations.
for c in ["my_dist_m", "my_dist_km", "opp_dist_m", "opp_dist_km", "dist_delta_km", "my_score", "opp_score"]:
    if c in rounds.columns:
        rounds[c] = pd.to_numeric(rounds[c], errors="coerce")

for c in ["my_avg_dist_m", "my_avg_dist_km", "opp_avg_dist_m", "opp_avg_dist_km"]:
    if c in games.columns:
        games[c] = pd.to_numeric(games[c], errors="coerce")

rounds["round_start"] = pd.to_datetime(rounds.get("round_start"), utc=True, errors="coerce")
rounds["round_end"] = pd.to_datetime(rounds.get("round_end"), utc=True, errors="coerce")
games["start_time"] = pd.to_datetime(games.get("start_time"), utc=True, errors="coerce")
games["end_time"] = pd.to_datetime(games.get("end_time"), utc=True, errors="coerce")

# Ensure guessed_correctly is boolean-like.
if "guessed_correctly" in rounds.columns:
    rounds["guessed_correctly"] = rounds["guessed_correctly"].map({"True": True, "False": False, True: True, False: False})

# Build distance delta if missing.
if "dist_delta_km" not in rounds.columns and {"my_dist_km", "opp_dist_km"}.issubset(rounds.columns):
    rounds["dist_delta_km"] = rounds["my_dist_km"] - rounds["opp_dist_km"]

print(f"rounds shape: {rounds.shape}")
print(f"games shape : {games.shape}")
print("game modes :", sorted(games["game_mode"].dropna().unique().tolist()) if "game_mode" in games.columns else "N/A")

rounds shape: (3293, 38)
games shape : (547, 13)
game modes : ['NmpzDuels', 'NoMoveDuels', 'StandardDuels']


In [64]:
valid_games = games[games["result"].isin(["W", "L"])].copy() if "result" in games.columns else games.copy()
total_games = len(valid_games)
wins = (valid_games["result"] == "W").sum() if "result" in valid_games.columns else np.nan
losses = (valid_games["result"] == "L").sum() if "result" in valid_games.columns else np.nan
wr = (wins / total_games * 100) if total_games else np.nan

print("=== Core KPIs ===")
print(f"Games (W/L only): {total_games}")
print(f"Wins: {wins} | Losses: {losses} | Win rate: {wr:.1f}%")
print(f"Total rounds: {len(rounds)}")
print(f"Countries seen: {rounds['real_country_name'].nunique(dropna=True) if 'real_country_name' in rounds.columns else 'N/A'}")

if "start_time" in valid_games.columns and valid_games["start_time"].notna().any():
    print("Date range:", valid_games["start_time"].min(), "->", valid_games["start_time"].max())

=== Core KPIs ===
Games (W/L only): 545
Wins: 324 | Losses: 221 | Win rate: 59.4%
Total rounds: 3293
Countries seen: 114
Date range: 2025-11-15 13:26:17.401000+00:00 -> 2026-04-21 22:33:23.379000+00:00


In [65]:
if {"game_mode", "result"}.issubset(games.columns):
    mode_plot = games[games["result"].isin(["W", "L"])].copy()
    mode_stats = (
        mode_plot.groupby("game_mode")["result"]
        .apply(lambda s: (s == "W").mean() * 100)
        .reset_index(name="win_rate_pct")
        .sort_values("win_rate_pct", ascending=False)
    )

    fig = px.bar(
        mode_stats,
        x="game_mode",
        y="win_rate_pct",
        color="win_rate_pct",
        color_continuous_scale=["#f87171", "#facc15", "#4ade80"],
        range_color=[0, 100],
        text=mode_stats["win_rate_pct"].round(1),
        title="Win Rate by Duel Mode",
    )
    fig.update_traces(texttemplate="%{text}%", textposition="outside")
    fig.update_layout(coloraxis_showscale=False, yaxis_title="Win rate %", xaxis_title="Mode")
    fig.show()

In [66]:
if {"start_time", "result"}.issubset(games.columns):
    ts = games[games["result"].isin(["W", "L"])].copy()
    ts = ts.dropna(subset=["start_time"]).sort_values("start_time")
    ts["win"] = (ts["result"] == "W").astype(int)
    ts["rolling_wr_20"] = ts["win"].rolling(20, min_periods=5).mean() * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=ts["start_time"],
        y=ts["rolling_wr_20"],
        mode="lines",
        line=dict(color="#60a5fa", width=2.5),
        name="Rolling WR (20 games)"
    ))
    fig.add_hline(y=50, line_width=1, line_dash="dash", line_color="#9ca3af")
    fig.update_layout(
        title="Rolling Win Rate Trend (20-game window)",
        yaxis_title="Win rate %",
        xaxis_title="Date",
        yaxis=dict(range=[0, 100]),
    )
    fig.show()

/usr/lib/python3/dist-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [67]:
if {"real_country_name", "guessed_correctly", "my_dist_km", "dist_delta_km"}.issubset(rounds.columns):
    country_stats = (
        rounds.groupby("real_country_name", dropna=True).agg(
            rounds=("guessed_correctly", "count"),
            correct=("guessed_correctly", "sum"),
            avg_dist_km=("my_dist_km", "mean"),
            avg_delta_km=("dist_delta_km", "mean"),
        ).reset_index()
    )
    country_stats["accuracy_pct"] = (country_stats["correct"] / country_stats["rounds"] * 100).round(1)
    country_stats = country_stats[country_stats["rounds"] >= 5].copy()

    top_acc = country_stats.sort_values("accuracy_pct", ascending=False).head(15)
    low_acc = country_stats.sort_values("accuracy_pct", ascending=True).head(15)

    fig_top = px.bar(
        top_acc.sort_values("accuracy_pct"),
        x="accuracy_pct", y="real_country_name", orientation="h",
        text="accuracy_pct", title="Top 15 Countries by Accuracy (min 5 rounds)",
        color="accuracy_pct", color_continuous_scale=["#facc15", "#4ade80"], range_color=[0, 100]
    )
    fig_top.update_traces(texttemplate="%{text}%", textposition="outside")
    fig_top.update_layout(coloraxis_showscale=False, xaxis_title="Accuracy %", yaxis_title="Country")
    fig_top.show()

    fig_low = px.bar(
        low_acc.sort_values("accuracy_pct", ascending=False),
        x="accuracy_pct", y="real_country_name", orientation="h",
        text="accuracy_pct", title="Bottom 15 Countries by Accuracy (min 5 rounds)",
        color="accuracy_pct", color_continuous_scale=["#f87171", "#facc15"], range_color=[0, 100]
    )
    fig_low.update_traces(texttemplate="%{text}%", textposition="outside")
    fig_low.update_layout(coloraxis_showscale=False, xaxis_title="Accuracy %", yaxis_title="Country")
    fig_low.show()

In [68]:
if {"real_country_name", "my_dist_km", "dist_delta_km", "guessed_correctly"}.issubset(rounds.columns):
    by_country = (
        rounds.groupby("real_country_name", dropna=True).agg(
            rounds=("guessed_correctly", "count"),
            avg_dist_km=("my_dist_km", "mean"),
            avg_delta_km=("dist_delta_km", "mean"),
        ).reset_index()
    )
    by_country = by_country[by_country["rounds"] >= 5].copy()
    by_country["avg_dist_km"] = by_country["avg_dist_km"].round(1)
    by_country["avg_delta_km"] = by_country["avg_delta_km"].round(1)

    fig = px.scatter(
        by_country,
        x="avg_dist_km", y="avg_delta_km",
        size="rounds", hover_name="real_country_name",
        color="avg_delta_km", color_continuous_scale=["#4ade80", "#facc15", "#f87171"],
        title="Country Profile: Avg Distance vs Avg Delta (size = rounds)",
        labels={"avg_dist_km": "Your avg distance (km)", "avg_delta_km": "Delta vs opponent (km)"}
    )
    fig.add_hline(y=0, line_width=1, line_dash="dash", line_color="#9ca3af")
    fig.update_layout(coloraxis_colorbar_title="Delta")
    fig.show()

In [ ]:
if {"real_country_name", "my_guessed_country_name", "guessed_correctly"}.issubset(rounds.columns):
    wrong = rounds[rounds["guessed_correctly"] == False].copy()
    conf = (
        wrong.groupby(["real_country_name", "my_guessed_country_name"]).size()
        .reset_index(name="count")
        .sort_values(["count", "real_country_name", "my_guessed_country_name"], ascending=[False, True, True])
    )

    top_pairs = conf.head(15).copy()
    focus_pair = conf[
        (conf["real_country_name"].str.lower() == "botswana")
        & (conf["my_guessed_country_name"].str.lower() == "south africa")
    ]
    if len(focus_pair) and not (
        (top_pairs["real_country_name"].str.lower() == "botswana")
        & (top_pairs["my_guessed_country_name"].str.lower() == "south africa")
    ).any():
        top_pairs = pd.concat([top_pairs.head(14), focus_pair.head(1)], ignore_index=True)
        top_pairs = top_pairs.sort_values(["count", "real_country_name", "my_guessed_country_name"], ascending=[False, True, True])

    if len(top_pairs):
        top_pairs["pair_label"] = top_pairs.apply(
            lambda r: f"{r['real_country_name']} -> {r['my_guessed_country_name']}", axis=1
        )
        fig = px.bar(
            top_pairs.sort_values("count"),
            x="count",
            y="pair_label",
            orientation="h",
            title="Top 15 Confusions (Real Country -> Guessed Country)",
            labels={"count": "Count", "pair_label": "Confusion Pair"},
            color="count",
            color_continuous_scale=["#facc15", "#f87171"],
        )
        fig.update_layout(coloraxis_showscale=False)
        fig.show()

In [70]:
if {"real_country_name", "guessed_correctly", "my_dist_km", "dist_delta_km"}.issubset(rounds.columns):
    country_metrics = (
        rounds.groupby("real_country_name", dropna=True).agg(
            rounds=("guessed_correctly", "count"),
            correct=("guessed_correctly", "sum"),
            avg_dist_km=("my_dist_km", "mean"),
            avg_delta_km=("dist_delta_km", "mean"),
        ).reset_index()
    )
    country_metrics["accuracy_pct"] = (country_metrics["correct"] / country_metrics["rounds"] * 100).fillna(0)

    def minmax_0_100(s):
        s = pd.to_numeric(s, errors="coerce")
        lo, hi = s.min(), s.max()
        if pd.isna(lo) or pd.isna(hi) or hi == lo:
            return pd.Series(50.0, index=s.index)
        return ((s - lo) / (hi - lo) * 100).clip(0, 100)

    train_base = country_metrics[country_metrics["rounds"] > 5].copy()
    train_base["rounds_imp"] = minmax_0_100(train_base["rounds"])
    train_base["delta_imp"] = minmax_0_100(train_base["avg_delta_km"])
    train_base["distance_imp"] = minmax_0_100(train_base["avg_dist_km"])
    train_base["accuracy_imp"] = (100 - train_base["accuracy_pct"]).clip(0, 100)
    train_base["train_priority_score"] = (
        0.40 * train_base["rounds_imp"]
        + 0.55 * train_base["delta_imp"]
        + 0.02 * train_base["distance_imp"]
        + 0.03 * train_base["accuracy_imp"]
    ).round(2)

    # Merge score back so all-country interactive chart can display score where available.
    country_metrics = country_metrics.merge(
        train_base[["real_country_name", "train_priority_score"]],
        on="real_country_name",
        how="left",
    )

    top_train = train_base.sort_values("train_priority_score", ascending=False).head(15)
    fig = px.bar(
        top_train.sort_values("train_priority_score"),
        x="train_priority_score",
        y="real_country_name",
        orientation="h",
        text="train_priority_score",
        title="Top 15 Training Priorities",
        color="train_priority_score",
        color_continuous_scale=["#facc15", "#f97316", "#ef4444"],
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(coloraxis_showscale=False, xaxis_title="Training priority score", yaxis_title="Country")
    fig.show()

In [ ]:
if "country_metrics" in locals() and len(country_metrics):
    full = country_metrics.sort_values("accuracy_pct", ascending=True).copy()
    fig_acc_all = px.bar(
        full,
        x="accuracy_pct",
        y="real_country_name",
        orientation="h",
        color="accuracy_pct",
        color_continuous_scale=["#f87171", "#facc15", "#4ade80"],
        range_color=[0, 100],
        title="All Countries: Accuracy % (Interactive)",
        hover_data={"rounds": True, "avg_dist_km": ":.1f", "avg_delta_km": ":.1f"},
    )
    fig_acc_all.update_layout(
        coloraxis_showscale=False,
        xaxis_title="Accuracy %",
        yaxis_title="Country",
        height=max(600, 22 * len(full)),
    )
    fig_acc_all.show()

    full_delta = country_metrics.sort_values("avg_delta_km", ascending=False).copy()
    fig_delta_all = px.bar(
        full_delta,
        x="avg_delta_km",
        y="real_country_name",
        orientation="h",
        color="avg_delta_km",
        color_continuous_scale=["#4ade80", "#facc15", "#f87171"],
        title="All Countries: Avg Delta vs Opponent (Interactive)",
        hover_data={"rounds": True, "accuracy_pct": ":.1f", "avg_dist_km": ":.1f"},
    )
    fig_delta_all.add_vline(x=0, line_width=1, line_dash="dash", line_color="#9ca3af")
    fig_delta_all.update_layout(
        coloraxis_showscale=False,
        xaxis_title="Average delta (km)",
        yaxis_title="Country",
        height=max(600, 22 * len(full_delta)),
    )
    fig_delta_all.show()

    full_train = country_metrics.dropna(subset=["train_priority_score"]).sort_values("train_priority_score", ascending=True)
    fig_train_all = px.bar(
        full_train,
        x="train_priority_score",
        y="real_country_name",
        orientation="h",
        color="train_priority_score",
        color_continuous_scale=["#facc15", "#f97316", "#ef4444"],
        title="All Countries (>5 rounds): Training Priority Score (Interactive)",
        hover_data={"rounds": True, "accuracy_pct": ":.1f", "avg_delta_km": ":.1f"},
    )
    fig_train_all.update_layout(
        coloraxis_showscale=False,
        xaxis_title="Training priority score",
        yaxis_title="Country",
        height=max(600, 22 * len(full_train)),
    )
    fig_train_all.show()

In [ ]:
if "country_metrics" in locals() and len(country_metrics):
    cm = country_metrics.copy()
    cm = cm[cm["rounds"] > 5].copy()

    top15_acc = cm.nlargest(15, "accuracy_pct").sort_values("accuracy_pct")
    bot15_acc = cm.nsmallest(15, "accuracy_pct").sort_values("accuracy_pct", ascending=False)

    top15_delta = cm.nsmallest(15, "avg_delta_km").sort_values("avg_delta_km", ascending=False)
    bot15_delta = cm.nlargest(15, "avg_delta_km").sort_values("avg_delta_km")

    cm_train = cm.dropna(subset=["train_priority_score"]).copy()
    top15_train = cm_train.nlargest(15, "train_priority_score").sort_values("train_priority_score")
    bot15_train = cm_train.nsmallest(15, "train_priority_score").sort_values("train_priority_score", ascending=False)

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            "Top 15 Accuracy", "Bottom 15 Accuracy",
            "Best 15 Delta (more negative)", "Worst 15 Delta (more positive)",
            "Top 15 Training Priority", "Bottom 15 Training Priority"
        ),
        horizontal_spacing=0.12,
        vertical_spacing=0.08,
    )

    fig.add_trace(go.Bar(x=top15_acc["accuracy_pct"], y=top15_acc["real_country_name"], orientation="h", marker_color="#22c55e", showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=bot15_acc["accuracy_pct"], y=bot15_acc["real_country_name"], orientation="h", marker_color="#ef4444", showlegend=False), row=1, col=2)

    fig.add_trace(go.Bar(x=top15_delta["avg_delta_km"], y=top15_delta["real_country_name"], orientation="h", marker_color="#22c55e", showlegend=False), row=2, col=1)
    fig.add_trace(go.Bar(x=bot15_delta["avg_delta_km"], y=bot15_delta["real_country_name"], orientation="h", marker_color="#ef4444", showlegend=False), row=2, col=2)

    fig.add_trace(go.Bar(x=top15_train["train_priority_score"], y=top15_train["real_country_name"], orientation="h", marker_color="#ef4444", showlegend=False), row=3, col=1)
    fig.add_trace(go.Bar(x=bot15_train["train_priority_score"], y=bot15_train["real_country_name"], orientation="h", marker_color="#22c55e", showlegend=False), row=3, col=2)

    fig.update_layout(height=1450, title_text="Top/Bottom 15 by Accuracy, Delta, and Training Score")
    fig.update_xaxes(title_text="Accuracy %", row=1, col=1)
    fig.update_xaxes(title_text="Accuracy %", row=1, col=2)
    fig.update_xaxes(title_text="Avg Delta (km)", row=2, col=1)
    fig.update_xaxes(title_text="Avg Delta (km)", row=2, col=2)
    fig.update_xaxes(title_text="Training Score", row=3, col=1)
    fig.update_xaxes(title_text="Training Score", row=3, col=2)
    fig.show()

In [73]:
# Outlier view: countries with fewer than 5 rounds.
if "country_metrics" in locals() and len(country_metrics):
    outliers = country_metrics[country_metrics["rounds"] < 5].copy()
    outliers = outliers.sort_values("rounds", ascending=True)

    if len(outliers):
        fig_out_acc = px.bar(
            outliers.sort_values("accuracy_pct", ascending=True),
            x="accuracy_pct",
            y="real_country_name",
            orientation="h",
            color="accuracy_pct",
            color_continuous_scale=["#f87171", "#facc15", "#4ade80"],
            range_color=[0, 100],
            title="Outlier Countries (<5 rounds): Accuracy %",
            hover_data={"rounds": True, "avg_delta_km": ":.1f", "avg_dist_km": ":.1f"},
        )
        fig_out_acc.update_layout(coloraxis_showscale=False, height=max(500, 22 * len(outliers)))
        fig_out_acc.show()

        fig_out_delta = px.bar(
            outliers.sort_values("avg_delta_km", ascending=False),
            x="avg_delta_km",
            y="real_country_name",
            orientation="h",
            color="avg_delta_km",
            color_continuous_scale=["#4ade80", "#facc15", "#f87171"],
            title="Outlier Countries (<5 rounds): Avg Delta",
            hover_data={"rounds": True, "accuracy_pct": ":.1f", "avg_dist_km": ":.1f"},
        )
        fig_out_delta.add_vline(x=0, line_width=1, line_dash="dash", line_color="#9ca3af")
        fig_out_delta.update_layout(coloraxis_showscale=False, height=max(500, 22 * len(outliers)))
        fig_out_delta.show()

In [74]:
cs  = pd.read_csv("https://raw.githubusercontent.com/pedrosalgado22/Live_Geoguessr_Dashboard/main/country_summary.csv")
tr  = pd.read_csv("https://raw.githubusercontent.com/pedrosalgado22/Live_Geoguessr_Dashboard/main/country_training_ranking.csv")
rnd = pd.read_csv("https://raw.githubusercontent.com/pedrosalgado22/Live_Geoguessr_Dashboard/main/roundsfinal.csv")
 
 
def name_to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None
 
# manual overrides for pycountry misses
OVERRIDES = {
    "Bolivia, Plurinational State of": "BOL",
    "Iran, Islamic Republic of": "IRN",
    "Korea, Republic of": "KOR",
    "Korea, Democratic People's Republic of": "PRK",
    "Taiwan, Province of China": "TWN",
    "Tanzania, United Republic of": "TZA",
    "Venezuela, Bolivarian Republic of": "VEN",
    "Viet Nam": "VNM",
    "Syrian Arab Republic": "SYR",
    "Russian Federation": "RUS",
    "Moldova, Republic of": "MDA",
    "Lao People's Democratic Republic": "LAO",
    "Congo, The Democratic Republic of the": "COD",
    "Micronesia, Federated States of": "FSM",
    "Palestine, State of": "PSE",
    "Czechia": "CZE",
    "United Kingdom": "GBR",
    "United States": "USA",
}
 
def get_iso3(name):
    if name in OVERRIDES:
        return OVERRIDES[name]
    return name_to_iso3(name)
 
CONTINENT = {
    "Africa": [
        "DZA","AGO","BEN","BWA","BFA","BDI","CPV","CMR","CAF","TCD","COM","COD","COG",
        "CIV","DJI","EGY","GNQ","ERI","SWZ","ETH","GAB","GMB","GHA","GIN","GNB","KEN",
        "LSO","LBR","LBY","MDG","MWI","MLI","MRT","MUS","MAR","MOZ","NAM","NER","NGA",
        "RWA","STP","SEN","SLE","SOM","ZAF","SSD","SDN","TZA","TGO","TUN","UGA","ZMB","ZWE",
    ],
    "Asia": [
        "AFG","ARM","AZE","BHR","BGD","BTN","BRN","KHM","CHN","CYP","GEO","IND","IDN",
        "IRN","IRQ","ISR","JPN","JOR","KAZ","KWT","KGZ","LAO","LBN","MYS","MDV","MNG",
        "MMR","NPL","PRK","OMN","PAK","PSE","PHL","QAT","SAU","SGP","KOR","LKA","SYR",
        "TWN","TJK","THA","TLS","TUR","TKM","ARE","UZB","VNM","YEM",
    ],
    "Europe": [
        "ALB","AND","AUT","BLR","BEL","BIH","BGR","HRV","CYP","CZE","DNK","EST","FIN",
        "FRA","DEU","GRC","HUN","ISL","IRL","ITA","XKX","LVA","LIE","LTU","LUX","MLT",
        "MDA","MCO","MNE","NLD","MKD","NOR","POL","PRT","ROU","RUS","SMR","SRB","SVK",
        "SVN","ESP","SWE","CHE","UKR","GBR","VAT",
    ],
    "North America": [
        "ATG","BHS","BRB","BLZ","CAN","CRI","CUB","DMA","DOM","SLV","GRD","GTM","HTI",
        "HND","JAM","MEX","NIC","PAN","KNA","LCA","VCT","TTO","USA",
    ],
    "South America": [
        "ARG","BOL","BRA","CHL","COL","ECU","GUY","PRY","PER","SUR","URY","VEN",
    ],
    "Oceania": [
        "AUS","FJI","KIR","MHL","FSM","NRU","NZL","PLW","PNG","WSM","SLB","TON","TUV","VUT",
    ],
    "Africa": [
        "DZA","AGO","BEN","BWA","BFA","BDI","CPV","CMR","CAF","TCD","COM","COD","COG",
        "CIV","DJI","EGY","GNQ","ERI","SWZ","ETH","GAB","GMB","GHA","GIN","GNB","KEN",
        "LSO","LBR","LBY","MDG","MWI","MLI","MRT","MUS","MAR","MOZ","NAM","NER","NGA",
        "RWA","STP","SEN","SLE","SOM","ZAF","SSD","SDN","TZA","TGO","TUN","UGA","ZMB","ZWE",
    ],
}
 
iso3_to_continent = {}
for continent, codes in CONTINENT.items():
    for c in codes:
        iso3_to_continent[c] = continent
 
df = tr.merge(
    cs[["real_country_name","avg_delta_km","accuracy_pct","avg_dist_km","biggest_confusion","worst_region"]],
    on="real_country_name", suffixes=("","_cs")
)
 
df["iso3"]      = df["real_country_name"].apply(get_iso3)
df["continent"] = df["iso3"].map(iso3_to_continent).fillna("Other")
 
df["delta_z"] = (df["avg_delta_km"] - df["avg_delta_km"].mean()) / df["avg_delta_km"].std()
 
export_cols = ["real_country_name","iso3","continent","avg_delta_km","delta_z",
               "train_priority_score","train_bucket","rounds","accuracy_pct"]

 
print(f"\nMaster table: {len(df)} countries")
df[export_cols].sort_values("delta_z", ascending=False).head(10)



Master table: 82 countries


,real_country_name,iso3,continent,avg_delta_km,delta_z,train_priority_score,train_bucket,rounds,accuracy_pct
1,India,IND,Asia,1581.528387,2.447670,69.96,Very High,62,67.741935
6,Oman,OMN,Asia,1280.255000,2.057565,54.45,Very High,12,41.666667
7,"Korea, Republic of",KOR,Asia,1149.931250,1.888815,53.78,Very High,17,58.823529
5,Cambodia,KHM,Asia,1149.757500,1.888590,55.55,Very High,24,45.833333
10,Viet Nam,VNM,Asia,1045.999524,1.754239,51.83,Very High,21,76.190476
9,Panama,PAN,North America,964.743636,1.649024,52.26,Very High,22,40.909091
3,Colombia,COL,South America,757.313390,1.380432,57.52,Very High,59,54.237288
18,Ireland,IRL,Europe,603.075185,1.180717,46.46,High,27,59.259259
11,Türkiye,TUR,Asia,579.190625,1.149790,51.23,Very High,48,62.500000
14,Poland,POL,Europe,521.288611,1.074815,47.57,Very High,36,55.555556


In [75]:
fig = px.choropleth(
    df,
    locations="iso3",
    color="delta_z",
    hover_name="real_country_name",
    hover_data={
        "delta_z": ":.2f",
        "avg_delta_km": ":.1f",
        "accuracy_pct": ":.1f",
        "rounds": True,
        "train_bucket": True,
        "iso3": False,
    },
    color_continuous_scale=["#4ade80", "#facc15", "#f87171"],
    range_color=[-2, 2],
    labels={
        "delta_z": "Delta Z-Score",
        "avg_delta_km": "Avg Delta (km)",
        "accuracy_pct": "Accuracy %",
        "rounds": "Rounds Played",
        "train_bucket": "Priority",
    },
    title="Distance Delta vs Opponent - World Map (Z-Score)"
)

fig.update_layout(
    paper_bgcolor="#0f0d3d",
    plot_bgcolor="#0f0d3d",
    font=dict(color="white", family="Inter"),
    title=dict(font=dict(size=18, color="white"), x=0.01),
    geo=dict(
        bgcolor="#0f0d3d",
        showframe=False,
        showcoastlines=True,
        coastlinecolor="rgba(255,255,255,0.15)",
        showland=True,
        landcolor="rgba(255,255,255,0.04)",
        showocean=True,
        oceancolor="#0f0d3d",
        showlakes=False,
        showcountries=True,
        countrycolor="rgba(255,255,255,0.08)",
        projection_type="natural earth",
    ),
    coloraxis_colorbar=dict(
        title="Delta Z",
        tickfont=dict(color="white"),
        titlefont=dict(color="white"),
        bgcolor="rgba(255,255,255,0.05)",
        bordercolor="rgba(255,255,255,0.1)",
    ),
    margin=dict(t=50, b=0, l=0, r=0),
    height=550,
)

fig.show()


fig2 = px.choropleth(
    df,
    locations="iso3",
    color="train_priority_score",
    hover_name="real_country_name",
    hover_data={
        "train_priority_score": ":.1f",
        "avg_delta_km": ":.1f",
        "accuracy_pct": ":.1f",
        "rounds": True,
        "train_bucket": True,
        "iso3": False,
    },
    color_continuous_scale=["#4ade80", "#facc15", "#f87171"],
    range_color=[0, 100],
    labels={
        "train_priority_score": "Training Score",
        "avg_delta_km": "Avg Delta (km)",
        "accuracy_pct": "Accuracy %",
        "rounds": "Rounds Played",
        "train_bucket": "Priority",
    },
    title="Training Priority Score - World Map"
)

fig2.update_layout(
    paper_bgcolor="#0f0d3d",
    plot_bgcolor="#0f0d3d",
    font=dict(color="white", family="Inter"),
    title=dict(font=dict(size=18, color="white"), x=0.01),
    geo=dict(
        bgcolor="#0f0d3d",
        showframe=False,
        showcoastlines=True,
        coastlinecolor="rgba(255,255,255,0.15)",
        showland=True,
        landcolor="rgba(255,255,255,0.04)",
        showocean=True,
        oceancolor="#0f0d3d",
        showlakes=False,
        showcountries=True,
        countrycolor="rgba(255,255,255,0.08)",
        projection_type="natural earth",
    ),
    coloraxis_colorbar=dict(
        title="Score",
        tickfont=dict(color="white"),
        titlefont=dict(color="white"),
        bgcolor="rgba(255,255,255,0.05)",
        bordercolor="rgba(255,255,255,0.1)",
    ),
    margin=dict(t=50, b=0, l=0, r=0),
    height=550,
)

fig2.show()


for continent in sorted(df["continent"].unique()):
    sub = df[df["continent"] == continent].sort_values("delta_z", ascending=False)
    if len(sub) < 2:
        continue

    colors = sub["delta_z"].apply(
        lambda v: "#4ade80" if v < -0.5 else "#facc15" if v < 0.5 else "#f87171"
    )

    fig_c = go.Figure(go.Bar(
        x=sub["delta_z"],
        y=sub["real_country_name"],
        orientation="h",
        marker_color=colors,
        text=sub["avg_delta_km"].round(1).astype(str) + " km",
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Delta Z: %{x:.2f}<br>"
            "Avg Delta: %{text}<br>"
            "Rounds: " + sub["rounds"].astype(str) + "<extra></extra>"
        ),
    ))

    fig_c.add_vline(x=0, line_width=1, line_color="rgba(255,255,255,0.3)")

    fig_c.update_layout(
        title=dict(text=f"{continent} - Distance Delta Z-Score", font=dict(size=16, color="white"), x=0.01),
        paper_bgcolor="#0f0d3d",
        plot_bgcolor="#0f0d3d",
        font=dict(color="white", family="Inter"),
        xaxis=dict(showgrid=False, zeroline=False, title="Delta Z-Score"),
        yaxis=dict(showgrid=False),
        height=max(300, len(sub) * 30),
        margin=dict(t=45, b=20, l=10, r=80),
    )

    fig_c.show()


fig3 = px.scatter(
    df,
    x="avg_delta_km",
    y="accuracy_pct",
    size="rounds",
    color="train_priority_score",
    hover_name="real_country_name",
    color_continuous_scale=["#4ade80", "#facc15", "#f87171"],
    range_color=[0, 100],
    size_max=40,
    labels={
        "avg_delta_km": "Avg Delta vs Opponent (km)",
        "accuracy_pct": "Country Guess Accuracy (%)",
        "rounds": "Rounds Played",
        "train_priority_score": "Training Priority",
    },
    title="Accuracy vs Delta - Bubble size = rounds played"
)

fig3.add_vline(x=0, line_width=1, line_dash="dash", line_color="rgba(255,255,255,0.3)")
fig3.add_hline(y=50, line_width=1, line_dash="dash", line_color="rgba(255,255,255,0.3)")

fig3.update_layout(
    paper_bgcolor="#0f0d3d",
    plot_bgcolor="#0f0d3d",
    font=dict(color="white", family="Inter"),
    title=dict(font=dict(size=16, color="white"), x=0.01),
    xaxis=dict(showgrid=False, color="rgba(255,255,255,0.4)"),
    yaxis=dict(showgrid=False, color="rgba(255,255,255,0.4)"),
    coloraxis_colorbar=dict(
        title="Priority",
        tickfont=dict(color="white"),
        titlefont=dict(color="white"),
    ),
    height=550,
    margin=dict(t=50, b=40, l=40, r=20),
)

fig3.show()